In [19]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv 
from langgraph.checkpoint.memory import InMemorySaver

In [20]:
load_dotenv()
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation"
)
model = ChatHuggingFace(llm=llm)


In [21]:
class joke(TypedDict):
    topic:str
    joke:str
    explanation:str

In [22]:
def generate(state:joke)->joke:
    prompt = f'generate a joke on the topic {state['topic']}'
    response  = model.invoke(prompt).content
    return {'joke':response}

In [23]:
def generate_explanation(state:joke)->joke:
    prompt = f'generate an explanation for the joke {state['joke']}'
    response  = model.invoke(prompt).content
    return {'explanation':response}

In [24]:
graph = StateGraph(joke)

graph.add_node('generate',generate)
graph.add_node('generate_explanation',generate_explanation)

graph.add_edge(START,'generate')
graph.add_edge('generate','generate_explanation')
graph.add_edge('generate_explanation',END)
checkointer = InMemorySaver()
chat = graph.compile()

In [25]:
initial_state = {'topic':'programming'}
config1 = {"config":{"thread_id":"1"}}
chat.invoke(initial_state, config1)

{'topic': 'programming',
 'joke': "Here's one:\n\nWhy do programmers prefer dark mode?\n\nBecause light attracts bugs.",
 'explanation': 'Here\'s an explanation for the joke:\n\nThe joke is a play on words, using a common phrase in a different context. In this case, the phrase "light attracts bugs" is a literal translation of the common idiom "light attracts bugs" (which is a colloquial way of saying "light attracts insects"), but it\'s applied to the world of programming.\n\nIn the context of programming, "bugs" refer to errors or faults in the code. So, the joke is saying that programmers prefer dark mode (a display setting that reduces the brightness of the screen) because the phrase "light attracts bugs" has a double meaning: in the literal sense, light attracts insects, but in the programming sense, light (or bright displays) attract bugs (or errors) in the code.\n\nThe humor comes from the unexpected twist on the familiar phrase, and the clever application of a common idiom to a 

In [26]:
chat.get_state(config1)

ValueError: No checkpointer set

In [14]:
list(chat.get_satate_history(config1))

AttributeError: 'CompiledStateGraph' object has no attribute 'get_satate_history'